# ocean: 1D

1D ocean time series data

```{dropdown}  **Branding suffix of datasets**
Datasets with the folloing branding suffix are shown in this page: \
[time]-[vertical]-[horizontal]-[domain]
```
* tavg-u-hm-sea
    - tavg: time average
    - u: ocean surface
    - hm: horizontal mean/sum
    - sea: ocean domain

In [ ]:
## Import libraries
import glob
import logging
import os
import sys
from os.path import join as pjoin

import matplotlib.pyplot as plt
import xarray as xr
from IPython.display import HTML, display
from pyprojroot import here
from xarray.coders import CFDatetimeCoder

from cmip7validate.utils import read_compound_names, read_variables

logging.basicConfig(
    filename="execute.log",
    level=logging.INFO,
    format="%(asctime)s %(message)s",
    force=True,
)
log = logging.getLogger("ocean_1D")
log.info("Starting ocean_1D.ipynb")

# Get project root
root_path = here()

In [ ]:
# load parameters
import yaml

with open(pjoin(root_path, 'params.yml'), 'r') as f:
    param = yaml.safe_load(f)
    cmorout = param['cmorout']
    source_id = param['source_id']
    experiment_id = param['experiment_id']
    variant_label = param['variant_label']
    version = param['version']

In [ ]:
# data path
data_path = os.path.join(cmorout, source_id, experiment_id, version)

# load methods for plotting and set defaults
methods =read_variables(pjoin(root_path, 'assets/methods.txt'))
#print(methods.keys())

---
```{dropdown}  **List of datasets:** 
List of compound names of datasets that needs to be cmorized. \
Those datasets which are not cmorized have no link.
```

In [ ]:
# load compound names
grid_label = 'g???'
coords = ('tavg-u-hm-sea')
cnames = read_compound_names('assets/variables.nml')
# examples of compound names:
# cnames = ['ocean.sos.tavg-u-hm-sea.mon.glb', 'ocean.masscello.tavg-u-hm-sea.mon.glb']
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]
    
    if realm != 'ocean' or coord not in coords:
        continue
    else:
        file_pattern = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}_*.nc"
        if not glob.glob(pjoin(data_path, file_pattern)):
            print(cname)
            continue
        else:
            display(HTML(f'<a href="#{cname}">{cname}</a>'))


---
```{dropdown}  **Dataset validated:**
Each validated dataset has some text information and some plots (2D and/or 1D)
```

In [ ]:

# loop through compound names and plot
for cname in cnames:
    realm = cname.split('.')[0]
    var = cname.split('.')[1]
    coord = cname.split('.')[2]
    freq = cname.split('.')[3]
    region = cname.split('.')[4]

    if realm != 'ocean' or coord not in coords:
        continue

    file_pattern = f"{var}_{coord}_{freq}_{region}_{grid_label}_{source_id}_{experiment_id}_{variant_label}_*.nc"
    data_files = sorted(glob.glob(os.path.join(data_path, file_pattern)))
    if data_files:
        log.info(f"{cname}")
        with xr.open_mfdataset(data_files, decode_times=CFDatetimeCoder(use_cftime=True)) as ds:
            if var in ds:
                data = ds[var]
            else:
                continue
    else:
        continue

    data1d = data
    display(HTML(f'<div id="{cname}"></div>'))
    print(f'\033[1m{cname}\033[0m')
    print(f'long name: {data1d.long_name} ({data1d.units})')
    print(f'original_name: {data1d.attrs["original_name"]} -> {var}')
    if 'history' in data1d.attrs:
        print(f'history: {data1d.attrs["history"]}')
    if 'comment' in data1d.attrs:
        print(f'comment: {data1d.attrs["comment"]}')

    fig = plt.figure(figsize=(16, 3), dpi=96)
    ax2 = fig.add_subplot(111)
    if data1d.sizes["time"] < 2:
        ax2.text(0.1, 0.8, (f"global value: {data1d.values[0]} {data1d.units}"))
        ax2.text(0.1, 0.6, ("less than 2 time steps."))
        ax2.text(0.1, 0.4, ("skipping timeseries plot."))
    else:
        data1d.plot(ax=ax2)
    plt.suptitle(data1d.long_name)

    plt.tight_layout()
    plt.show()

    del data, data1d
    del ax2, fig